**Version 0.2 · 2026-08-05 17:09 BRT · git tag `v0.2`**

# THUMS — energy closure in the wellbore model

`JaderBarbosaLSU/wellbore-thermal-storage` · `notebooks/01_energy_closure.ipynb`

The physics lives in the `thums_core` package, not in this notebook. This file is a case
study: set up, run, plot. See `docs/FIX_energy_closure.md` for the argument.

> Do **not** press Ctrl/Cmd+S — it forks a private copy into Drive that stops receiving
> updates. Do not edit code here; describe the change instead. After any update open a
> **new tab**, then *Runtime → Restart and run all*. Because the physics is an installed
> package, a re-run without a restart can silently use the previously installed version.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JaderBarbosaLSU/wellbore-thermal-storage/blob/main/notebooks/01_energy_closure.ipynb)

In [ ]:
!pip install -q git+https://github.com/JaderBarbosaLSU/wellbore-thermal-storage.git
import thums_core
print(thums_core.VERSION_STAMP)

## 1. The inconsistency

The fluid side uses the **finned** perimeter; the melt front solves a **bare cylinder**.
They are two routes to the same energy and they disagree.

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from thums_core.config import BASELINE
from thums_core import _legacy_physics as L, well_marched as W, sizing
from thums_core.system import _cycles

case = BASELINE
cy, pcm, geom, num = case.cycle, case.pcm, case.geometry, case.numerics
perim_fin  = 2*np.pi*geom.r_e + 2*geom.num_fins*geom.fin_L
perim_bare = 2*np.pi*geom.r_e
print(f'fluid side  perimeter : {perim_fin:.4f} m')
print(f'melt front  perimeter : {perim_bare:.4f} m')
print(f'ratio                 : {perim_fin/perim_bare:.2f}')

## 2. Case set-up

Every number comes from the `Case` object — no loose globals.

In [ ]:
rank, hp, T = _cycles(case)
Q_in_ORC   = (cy.W_dot_el_out/cy.Turb_eff/cy.ElG_eff)/rank['rank_eff']
D_E_out_HP = Q_in_ORC*cy.t_dc*3600.*(1+cy.loss_surplus)   # kJ
cp_w   = L.CP.PropsSI('C','T',0.5*(T['T_3c']+T['T_2c']),'P',cy.P,cy.fluid2)/1000.
m_tot  = D_E_out_HP/cy.t_ch/3600./cp_w/cy.DT_3C_2C

n_seg, N_lay = 60, case.N_lay
dTm   = cy.DT_3C_2C/N_lay
T_m_lay = (pcm.T_m - np.arange(N_lay, dtype=float)*dTm).tolist()
z_lay = np.linspace(0, geom.L_tube, N_lay+1)
z_c   = (np.arange(n_seg)+0.5)*geom.L_tube/n_seg
T_m_seg = np.array([T_m_lay[min(np.searchsorted(z_lay,z,side='right')-1, N_lay-1)] for z in z_c])
times = np.logspace(np.log10(1.), np.log10(cy.t_ch*3600.), 40)
print(f'COP {hp["hp_cop"]:.4f} | eta_ORC {rank["rank_eff"]:.4f} | m_dot {m_tot:.3f} kg/s | D_E {D_E_out_HP/1e6:.1f} GJ')

## 3. Closure: legacy front vs energy-balance front

`Q_in` is the heat the fluid gives up. `rho·h_m·V_melt` is the latent heat the front
absorbed. They must agree to within the neglected sensible term (~5 %).

In [ ]:
rows = []
for N in (12., 24.):
    for label, k_wall in (('legacy k_w=0.45', pcm.k_l), ('correct k_w=45', geom.k_wall)):
        m1 = m_tot/(N*geom.num_tubes)
        _,_,Vm,Qk = L.time_profiles_melt(times, geom.legacy_vector(), T['T_4c'], N_lay,
                        T_m_lay, k_wall, geom.Rf_i, m1, cy.P, cy.fluid2,
                        pcm.k_l, pcm.cp_l, pcm.rho_l, pcm.h_m, n_seg, num.delta_max)
        V_leg = Vm[times[-1]]
        cl_leg = abs(Qk*1e3 - pcm.rho_l*pcm.h_m*V_leg)/(Qk*1e3)
        r = W.march(geom, pcm, T_inlet=T['T_4c'], T_m_seg=T_m_seg, m_dot=m1, P=cy.P,
                    fluid=cy.fluid2, k_wall=k_wall, Rf_i=geom.Rf_i, times=times,
                    n_segments=n_seg, mode='charge')
        dz = geom.L_tube/n_seg
        rows.append((N, label, cl_leg, V_leg*geom.num_tubes/geom.V_well,
                     r.closure, r.state.volume(dz, geom.num_tubes)/geom.V_well))
print(f"{'N':>4} {'variant':<18} {'legacy err':>11} {'legacy eps':>11} {'marched err':>12} {'marched eps':>12}")
for N,l,c1,e1,c2,e2 in rows:
    print(f'{N:4.0f} {l:<18} {c1*100:10.1f}% {e1:11.3f} {c2:12.1e} {e2:12.3f}')

## 4. Sizing under both constraints

`N_wells = max(N_heat_transfer, N_inventory)`. The legacy code used only the first,
and printed the second as *Ideal number of wells* without ever applying it.

In [ ]:
for label, k_wall in (('legacy k_w=0.45', pcm.k_l), ('correct k_w=45', geom.k_wall)):
    s = sizing.size_well_field(geom=geom, pcm=pcm, cy=cy, num=num, T_inlet=T['T_4c'],
            T_m_seg=T_m_seg, m_dot_total=m_tot, D_E_required_kJ=D_E_out_HP,
            times=times, k_wall=k_wall, n_segments=n_seg)
    print(f'{label:18s} N={s.N_wells:6.2f}  (heat {s.N_heat:5.2f} | inventory {s.N_inventory:5.2f})  binding: {s.binding}  eps={s.eps_pcm:.3f}')

## 5. Melt-front profile along the well

In [ ]:
m1 = m_tot/(13.65*geom.num_tubes)
r  = W.march(geom, pcm, T_inlet=T['T_4c'], T_m_seg=T_m_seg, m_dot=m1, P=cy.P,
             fluid=cy.fluid2, k_wall=geom.k_wall, Rf_i=geom.Rf_i, times=times,
             n_segments=n_seg, mode='charge')
from thums_core.stefan import delta_from_area
d = delta_from_area(r.state.A_melt, geom.r_e)*1000
fig, ax = plt.subplots(figsize=(6,3.6))
ax.plot(z_c, d, lw=1.6)
ax.axhline(geom.D_well/2*1000 - geom.r_e*1000, ls='--', lw=1, color='crimson',
           label='front reaches borehole wall')
ax.set_xlabel('distance along tube  z [m]'); ax.set_ylabel(r'melt thickness $\delta$ [mm]')
ax.set_title(f'End of charging · {thums_core.VERSION_STAMP}', fontsize=9)
ax.legend(frameon=False, fontsize=8); fig.tight_layout(); plt.show()